# Mini Demo: Tokenizer and Transformer in a Tiny Language Model

This notebook file is mainly generated by **ChatGPT 5.1**. It demonstrates two core ideas behind large language models (LLMs):

1. **Tokenizer**: how we convert text → tokens → integer IDs → back to text.
2. **Transformer block**: how self‑attention and a small feed‑forward network
   turn token embeddings into contextualized representations.

We'll keep everything tiny and simple so you can see all the pieces.

## 0. Imports

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## 1. A Very Simple Tokenizer

Real LLMs use subword tokenizers like **BPE** or **SentencePiece**.
Here, we will use a **toy word‑level tokenizer** so the logic is easy to see:

1. Lowercase the text
2. Split on spaces → tokens
3. Build a vocabulary (token → integer ID)
4. Implement `encode` (text → IDs) and `decode` (IDs → text)

This is for **illustration**, not for production use.

In [ ]:
from collections import Counter  # count how many times an item shows up

# A tiny text corpus to build a vocabulary
text_corpus = [
    "the quick brown fox jumps over the lazy dog",
    "the quick blue fox jumps high",
    "a lazy dog sleeps",
]

def basic_tokenize(text: str):
    """Toy tokenizer: lowercase + split on spaces."""
    return text.lower().split()

cnt = 0
# Build vocabulary
counter = Counter()
for sent in text_corpus:
    counter.update(basic_tokenize(sent))
    print("sentence", cnt, '-->', sent)
    cnt += 1

print(" ")
print(counter)
print(" ")


# Special tokens
PAD = "<pad>"
UNK = "<unk>"

vocab = [PAD, UNK] + sorted(counter.keys())
stoi = {tok: i for i, tok in enumerate(vocab)}  # string → index (Python dictionary)
itos = {i: tok for tok, i in stoi.items()}      # index → string

print("Vocabulary:")
print(vocab)
print("\nToken → ID mapping (stoi):")
print(stoi)


### Encode / Decode

We now create `encode` and `decode` functions:

- `encode(text)`: text → tokens → integer IDs
- `decode(ids)`: integer IDs → tokens → text


In [ ]:
def encode(text: str):
    tokens = basic_tokenize(text)
    ids = [stoi.get(tok, stoi[UNK]) for tok in tokens]
    return ids

def decode(ids):
    tokens = [itos.get(i, UNK) for i in ids]
    return " ".join(tokens)

example = "the quick fox jumps"
ids = encode(example)

print("Text:   ", example)
print("Tokens: ", basic_tokenize(example))
print("IDs:    ", ids)
print("Back:   ", decode(ids))


## 2. From Tokens to Embeddings

LLMs do not work directly on integer IDs. Instead, each token ID is mapped
to a dense vector in ℝ^d (an **embedding**).

Here we create a tiny embedding layer and apply it to one sentence.

In [ ]:
# Hyperparameters for toy model
vocab_size = len(vocab)
d_model = 32  # embedding dimension

embedding = nn.Embedding(vocab_size, d_model)

sentence = "the quick brown fox jumps over the lazy dog"
ids = encode(sentence)
x = torch.tensor(ids).unsqueeze(0)  # [batch, seq_len] = [1, L]

print("Token IDs tensor shape:", x.shape)
emb = embedding(x)                  # [1, L, d_model]
print("Embedding shape:", emb.shape)


## 3. A Tiny Transformer Block

Next, we build a **single Transformer block** with:

1. LayerNorm → Multi‑head self‑attention → residual connection
2. LayerNorm → Feed‑forward network (FFN) → residual connection

This mirrors the structure used in real LLMs, just withmuch smaller dimensions and only one layer.

---
**Some explanations of the code.**
* `d_model` is the size of the embedding vector for each token.
    - every token is represented as a 32-dimensional vector
    - Q, K, V vectors will also have size 32 (unless projected smaller)
    - attention heads will each receive a portion of this dimension
      
    In a full LLM like GPT-3, d_model = 12288. In your tiny transformer, 32 is small and good for demonstration. 

* `n_heads`: This specifies how many parallel attention mechanisms you want.
    - the model splits the 32-dim vector into 4 heads
    - each head gets: $d_k = d_{model}/n_{heads} = 32/4 = 8$ values
    - each head learns different relationships (syntax, long-range connection, numbers, etc.)
    - So the attention actually computes:
        ```
        head_1: 8-d attention
        head_2: 8-d attention
        head_3: 8-d attention
        head_4: 8-d attention
        ```
        <br>
    Then the heads are concatenated back to a 32-dim output.<br>

* `d_ff` — Feed-Forward Network (FFN) hidden size
    - This is the size of the inner layer of the MLP (Multilayer Perceptron) inside the transformer block.
    - Typical transformer block structure:
        ```
        Self-Attention → LayerNorm → Feed Forward Network (FFN) → LayerNorm
        ```
<br>

* `LayerNorm`: Layer Normalization is a normalization technique used inside transformers (and LLMs) to stabilize training and improve performance. Normalizing the feature components to have 1) Stable training, 2) Smoother gradients, 3) Faster convergence, and 4) Better precision
* `Q, K, V`:
    * Q — Query Vector: For each token, Query represents what this token wants to find in other tokens.
    * K — Key Vector: Key represents what information this token offers to others.
    * V — Value Vector: Value is the actual information content passed on when a token is attended to.
---
---
> Imagine you're reading: “The battery electrode cracked because it was brittle.” <br>
> To understand "it", your brain looks for something relevant in the earlier sentence.
Attention does the same thing:
>
>> Query asks:
>> What am I looking for? <br>
>> Key answers:
>> Do I contain what you're looking for? <br>
>> Value provides:
>> Here’s the information you get from me.
> How Q, K, V work together
> 
> Given a sequence of tokens, we compute Q, K, and V for each one using learned weight matrices:
> $$\mathbf{Q}=\mathbf{X}\mathbf{W}_Q ,\mathbf{K}=\mathbf{X}\mathbf{W}_K, \mathbf{V}=\mathbf{X}\mathbf{W}_V$$
> $\mathbf{X}$ = embedding of tokens (shape [seq_len, d_model]), <br>
> $\mathbf{W}_Q, \mathbf{W}_K, \mathbf{W}_V$ = learned projection matrices.

* Attention Score Calculation:
    - To see how much token i should attend to token j:
  
      $$\text{score}(i,j) = Q_i \cdot K_j$$ 

  - Then, scale and softmax:

    $$a_{ij} = \text{softmax}\bigg(\frac{Q_i K_j}{\sqrt{d_k}}\bigg).$$
 
    where $d_k = d_{model}/n_{heads} = 32/4 = 8$.
  - Output for each token:

    $$\text{output}_i = \sum_{j} a_{ij} V_{j}.$$
 
    The output is a weighted sum of Values, where weights depend on how well Queries match Keys.
 
---
---

In [ ]:
class TinyTransformerBlock(nn.Module):
    def __init__(self, d_model=32, n_heads=4, d_ff=64):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        # Projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model) # d_model to d_model linear layer
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # Output projection
        self.W_o = nn.Linear(d_model, d_model)

        # Feed‑forward network
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )

        # Layer norms
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        """Forward pass through one Transformer block.

        x: [batch, seq_len, d_model]
        Returns: (x_out, attention_weights)
        """
        b, L, d = x.shape

        # ---- Multi‑head self‑attention ----
        # Pre‑norm
        h = self.ln1(x)

        Q = self.W_q(h)  # [b, L, d_model]
        K = self.W_k(h)
        V = self.W_v(h)

        # Reshape to [b, n_heads, L, d_head], view is reshape
        Q = Q.view(b, L, self.n_heads, self.d_head).transpose(1, 2)
        K = K.view(b, L, self.n_heads, self.d_head).transpose(1, 2)
        V = V.view(b, L, self.n_heads, self.d_head).transpose(1, 2)

        # Scaled dot‑product attention
        # scores: [b, n_heads, L, L]
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        attn = F.softmax(scores, dim=-1)

        # Context: [b, n_heads, L, d_head]
        context = attn @ V

        # Back to [b, L, d_model]
        context = context.transpose(1, 2).contiguous().view(b, L, d)

        # Output projection + residual
        x = x + self.W_o(context)

        # ---- Feed‑forward ----
        h2 = self.ln2(x)
        ff_out = self.ff(h2)

        # Residual again
        x = x + ff_out

        return x, attn


### Run the Transformer Block

We apply our `TinyTransformerBlock` to the embeddings.
The output has the same shape, but each position has now "seen" the
other tokens through self‑attention.

In [ ]:
block = TinyTransformerBlock(d_model=d_model, n_heads=4, d_ff=64)

x_in = emb              # [1, seq_len, d_model]
x_out, attn = block(x_in)

print("Input shape: ", x_in.shape)
print("Output shape:", x_out.shape)
print("Attention shape:", attn.shape)  # [b, n_heads, L, L]


## 4. Visualizing Self‑Attention

Each attention matrix has shape `[seq_len, seq_len]` for each head.

- Rows = **query** positions ("which token is looking")
- Columns = **key** positions ("which token is being looked at")
- Color = how strongly the query attends to the key


In [ ]:
# Take head 0 for visualization
tokens = basic_tokenize(sentence)
L = len(tokens)
attn_head0 = attn[0, 0].detach().numpy()  # [L, L]

plt.figure(figsize=(6, 5))
plt.imshow(attn_head0, interpolation="nearest")
plt.xticks(range(L), tokens, rotation=90)
plt.yticks(range(L), tokens)
plt.colorbar()
plt.title("Self‑attention weights (head 0)")
plt.tight_layout()
plt.show()


## 5. Summary

In this notebook, we built a **tiny LLM core** from scratch:

1. A **tokenizer** to map text ↔ tokens ↔ integer IDs.
2. An **embedding layer** to turn token IDs into dense vectors.
3. A **single Transformer block** with multi‑head self‑attention and a
   feed‑forward network.
4. A simple visualization of the **attention matrix**.

To turn this into a real language model, you would:

- Stack many Transformer blocks.
- Add positional encodings.
- Add an output layer that predicts the next token.
- Train on a very large text corpus.


---
## 6. Test a tiny langauge model.

Below, we build a tiny language model using the `TinyTransformerBlock` earlier.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=32, n_heads=4, d_ff=64, max_len=128):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_len = max_len

        # token & position embeddings
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        # a single transformer block (the one you already built)
        self.block = TinyTransformerBlock(d_model=d_model,
                                          n_heads=n_heads,
                                          d_ff=d_ff)

        # final layernorm + linear head to logits
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        """
        idx: [b, L] integer token ids
        returns: logits [b, L, vocab_size]
        """
        b, L = idx.shape
        device = idx.device

        # 1) embeddings
        tok = self.tok_emb(idx)  # [b, L, d_model]

        # 2) positional encodings
        positions = torch.arange(L, device=device).unsqueeze(0)  # [1, L]
        pos = self.pos_emb(positions)                            # [1, L, d_model]

        # 3) add them
        x = tok + pos                                            # [b, L, d_model]

        # 4) transformer block (ln → attn → residual, ln → MLP → residual)
        out = self.block(x)
    
        # if the block returns (x, attn_scores) or similar, unpack it:
        if isinstance(out, tuple):
            x, _ = out   # ignore attention scores
        else:
            x = out                                        # [b, L, d_model]

        # 5) final norm + logits over vocab
        x = self.ln_f(x)                                         # [b, L, d_model]
        logits = self.head(x)                                    # [b, L, vocab_size]
        return logits


**Read in the text_corpus to the LM.**

In [ ]:
vocab_size = len(stoi)
model = TinyTransformerLM(vocab_size=vocab_size, d_model=32, n_heads=4, d_ff=64)
print(vocab_size)

**Finally, we build a `generate` function to predict the words appearing after the prompt.**

In [ ]:
def generate(
    model,
    prompt,
    max_new_tokens=20,
    temperature=1.0,
    top_k=None,
    device="cpu"
):
    model.eval()
    model.to(device)

    # 1) encode prompt to token ids using your encode()
    input_ids = encode(prompt)
    input_ids = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)  # [1, L]

    for _ in range(max_new_tokens):
        # 2) forward pass
        with torch.no_grad():
            logits = model(input_ids)          # [1, L, vocab_size]

        # 3) take logits for the last position
        next_logits = logits[:, -1, :]         # [1, vocab_size]

        # 4) temperature scaling
        next_logits = next_logits / temperature

        # 5) (optional) top-k filtering
        if top_k is not None:
            values, _ = torch.topk(next_logits, top_k)
            min_values = values[:, -1].unsqueeze(-1)
            next_logits = torch.where(
                next_logits < min_values,
                torch.full_like(next_logits, float("-inf")),
                next_logits,
            )

        # 6) probabilities
        probs = F.softmax(next_logits, dim=-1)  # [1, vocab_size]

        # 7) sample next token (or use argmax for greedy)
        next_id = torch.multinomial(probs, num_samples=1)  # [1, 1]
        # next_id = torch.argmax(probs, dim=-1, keepdim=True)  # <- greedy alternative

        # 8) append to sequence
        input_ids = torch.cat([input_ids, next_id], dim=1)  # [1, L+1]

    # 9) decode back to text using your decode()
    out_tokens = input_ids[0].tolist()
    return decode(out_tokens)


**Give it a try!!**

In [ ]:
prompt = "Your are"
text = generate(model, prompt, max_new_tokens=10, temperature=0.8, top_k=10)
print(text)



---
### A webpage of illustrating the working process of transformers.

#### https://poloclub.github.io/transformer-explainer/